In [ ]:
# Data Handling and Infrastructure for AI Project

# Predicting NVD CVSS v3.1 Severity for Newly Published CVEs Using Machine Learning

## Milestone 1 — Data Source, ML Task and Storage Architecture

### Terminology

- **CVE (Common Vulnerabilities and Exposures):** a standard public identifier for a disclosed cybersecurity vulnerability.
- **CVSS (Common Vulnerability Scoring System):** a standard framework for assessing the severity of vulnerabilities.
- **NVD (National Vulnerability Database):** the NIST-maintained vulnerability database that enriches CVE records with vulnerability-management information, including CVSS assessments.
- **CNA (CVE Numbering Authority):** an organisation authorised by the CVE Program to assign and publish CVE records.
- **CWE (Common Weakness Enumeration):** a classification system for common software and hardware weakness types.
- **ML (Machine Learning):** methods that learn patterns from existing data in order to make predictions on new data.
- **TF-IDF (Term Frequency–Inverse Document Frequency):** a method for converting text into numerical features based on the relative importance of words.
- **API (Application Programming Interface):** an interface through which software systems can communicate.
- **CPE (Common Platform Enumeration):** a structured naming system for identifying software, hardware and operating-system products.
- **DVC (Data Version Control):** a tool for versioning datasets and machine-learning artifacts alongside source code.
- **MLOps (Machine Learning Operations):** practices for managing reproducible machine-learning training, deployment, monitoring and updates.

---

## 1. Project Overview

This project investigates whether information available when a CVE is published can be used to predict the severity that is later assigned to it by NVD using CVSS version 3.1.

The practical motivation is the period between the publication of a vulnerability and the availability of an NVD severity assessment. An internal review of the Edgescan codebase identified that a missing CVSS score results in fallback handling. This creates a realistic use case for an ML system that could provide a temporary severity estimate, clearly identified as a prediction rather than an official CVSS assessment.

The proposed system will ingest public vulnerability data, construct a leakage-controlled historical training dataset, train a four-class classifier, and eventually expose predictions through an API. The model will predict one of four severity categories:

**LOW, MEDIUM, HIGH or CRITICAL.**

The project is primarily a data-handling and infrastructure project rather than an attempt to design a complex ML model. Particular attention will therefore be given to data provenance, leakage prevention, reproducibility, storage, automated processing, model versioning and the later deployment/update lifecycle.

---

## 2. Data Sources and Label Construction

### 2.1 Dataset Description

Two related public data sources will be used.

The first is the **official CVE List**, distributed in CVE JSON 5 format by the CVE Program. It contains CVE records published by CNAs and includes fields such as the vulnerability description, publishing organisation, publication timestamps, references and affected-product information.

The second source is the **NVD**, maintained by the National Institute of Standards and Technology (NIST). NVD provides vulnerability data through APIs and JSON 2.0 data feeds.

The two sources will be joined using the CVE identifier, for example:

```text
CVE-2026-12345
```

The CVE List will primarily provide the historical information used as model input, while NVD will provide the eventual target label.

This separation is important because the intended task is to reproduce what would have been known around publication time rather than train a model using information that was added later.

### 2.2 Dataset Sources

#### CVE List

Official CVE List repository:

```text
CVEProject/cvelistV5
```

The repository publishes CVE records in CVE JSON 5 format. Historical Git revisions and release snapshots may also be useful for reconstructing the state of a CVE around its original publication time.

#### NVD

NVD JSON 2.0 vulnerability feeds provided by NIST will be used for the target labels.

### 2.3 Dataset Size

A complete 2026 NVD snapshot downloaded on **3 October 2026** contains **67,186 CVE records**.

Analysis of this dataset identified **11,454 CVEs** containing an NVD-authored Primary CVSS v3.1 assessment with one of the four target severity classes.

The current target distribution is:

| Severity | Records | Approx. Percentage |
|---|---:|---:|
| LOW | 192 | 1.7% |
| MEDIUM | 5,214 | 45.5% |
| HIGH | 4,213 | 36.8% |
| CRITICAL | 1,835 | 16.0% |
| **Total** | **11,454** | **100%** |

This provides a potential labelled dataset above the assignment requirement of **10,000 learning samples**.

The final usable dataset size may be slightly lower after the CVE List and NVD sources are joined and publication-time leakage controls are applied.

The number of usable samples will therefore be verified again after preprocessing. If the leakage-safe dataset falls below 10,000 records, the publication window will be extended backwards rather than weakening the target-label definition.

### 2.4 Label Construction

A major issue in the NVD data is that one CVE may contain multiple CVSS assessments from different sources.

To keep the target consistent, the project will use only a CVSS v3.1 metric satisfying the following conditions:

```text
source = "nvd@nist.gov"
type = "Primary"
cvssData.version = "3.1"
baseSeverity ∈ {LOW, MEDIUM, HIGH, CRITICAL}
```

The target variable will therefore be:

**NVD-authored Primary CVSS v3.1 base severity.**

CVSS assessments produced by CNAs or other organisations will not be used as alternative target labels.

This decision avoids mixing scoring decisions made by different organisations and gives the classification task a consistent interpretation.

### 2.5 Licensing and Permitted Use

Both datasets are publicly accessible.

The CVE List may be downloaded and used according to the CVE Program Terms of Use.

NVD is maintained by NIST as a public vulnerability-management data repository.

The project will retain source attribution and use the datasets only for the educational and research purposes described in this project.

### 2.6 Why These Datasets Were Selected

These datasets were selected for several reasons.

First, they represent the real public data flow associated with CVE publication and later vulnerability enrichment rather than an artificially prepared ML dataset.

Second, the data volume is large enough for the assignment. The current 2026 dataset contains over **67,000 CVE records** and more than **11,000 examples** with the selected NVD CVSS v3.1 target.

Third, the data contain realistic imperfections that make them particularly appropriate for a data-handling project. These include missing values, different CVSS versions, multiple assessments for one CVE, inconsistent record structures, rejected records, different publishing organisations and substantial variation in description style.

Finally, the data naturally support an ongoing production scenario. New CVEs continue to be published, records are updated over time, and labels can become available after the initial publication. This creates a realistic later retraining and model-update lifecycle.

### 2.7 Raw Data Characteristics and Expected Quality Problems

The raw data are semi-structured JSON records rather than a clean tabular ML dataset.

Initial inspection has identified several issues that must be addressed during later preprocessing:

- Some CVEs have no CVSS v3.1 assessment.
- Multiple CVSS versions may appear, including v2, v3.0, v3.1 and v4.0.
- A CVE can contain assessments from both NVD and other organisations.
- Different assessors can assign different scores or severity categories to the same CVE.
- Rejected CVE records are present.
- Vulnerability status varies between records.
- Publishing organisations use different structures and description styles.
- Vendor and product names may be inconsistent.
- Fields such as CWE or affected products may be missing.
- Descriptions from a single publisher can be highly templated or near-duplicates.
- Records can be updated after publication, creating a risk of using future information.

For example, the complete NVD snapshot contains records in states including:

```text
Analyzed
Deferred
Awaiting Analysis
Received
Modified
Rejected
Undergoing Analysis
```

These issues will provide the basis for the exploratory data analysis and repeatable data-cleaning pipeline developed in **Milestone 2**.

---

## 3. Machine Learning Task and Leakage Boundary

### 3.1 Problem Definition

The ML task is:

> Given the information that would realistically have been available when a CVE was published, predict the NVD CVSS v3.1 severity that is subsequently assigned to that vulnerability.

The intended prediction scenario is:

```text
CVE Published
      ↓
Publication-Time Information Available
      ↓
NVD v3.1 Assessment Not Yet Available
      ↓
ML Model Predicts Severity
      ↓
LOW / MEDIUM / HIGH / CRITICAL
```

The output is an ML prediction and will not be presented as an official CVSS assessment.

### 3.2 Target Variable

The target variable is:

```text
nvd_cvss_v31_severity
```

Possible values are:

- LOW
- MEDIUM
- HIGH
- CRITICAL

The numerical CVSS score between 0.0 and 10.0 will be retained for analysis where useful but will not initially be the prediction target.

### 3.3 Proposed Input Features

The initial candidate feature set includes:

#### Description Text

The English vulnerability description is expected to contain information about the vulnerability mechanism and potential impact.

This will initially be represented using **TF-IDF**.

#### Publishing Organisation / CNA

The organisation responsible for publishing the CVE may provide useful contextual information and also allows investigation of publisher-specific patterns.

#### Publication Date Information

Features derived from the publication timestamp may be considered where they are relevant and do not encode future information.

#### Reference Count

The number of references available in the publication-time record may be used as a simple metadata feature.

The following fields are considered **candidate features but will not automatically be included**:

- CWE
- Vendor
- Product
- Affected-version information

Their availability at the intended prediction time must first be validated.

### 3.4 Leakage Boundary

Preventing data leakage is a central requirement of the project.

Any information created or added as part of later NVD enrichment must not be used to predict the NVD target.

The following fields will therefore be excluded from the model inputs:

- Any NVD CVSS score
- CVSS vector components
- CVSS severity
- Later NVD enrichment
- Post-publication scoring information
- CISA or other later-added assessment fields
- Any other feature that cannot be shown to have existed at the intended prediction time

CPE data and CWE will not be assumed to be safe features until their historical availability has been checked.

Historical CVE List snapshots or Git history will be investigated to reconstruct the state of a CVE around publication rather than relying blindly on its current record.

This is necessary because a current CVE record may contain information that was added after its original publication.

### 3.5 Type of ML Task

This is a **supervised multiclass classification** task with four classes.

The initial useful model is expected to use:

```text
TF-IDF Description Features
        +
Selected Publication-Time Metadata
        ↓
Logistic Regression
        ↓
LOW / MEDIUM / HIGH / CRITICAL
```

A simple model is intentional.

The primary purpose of the project is to demonstrate a reliable end-to-end data and ML infrastructure rather than maximise model complexity.

---

## 4. Data Splitting and Evaluation Strategy

### 4.1 Training Set

The oldest portion of the eligible CVEs will be used for model training.

A provisional split of approximately:

- **70% training**
- **15% validation**
- **15% test**

will be investigated.

The exact calendar cut-offs will be chosen after examining the chronological distribution of the final cleaned dataset.

### 4.2 Development / Validation Set

The validation set will contain CVEs published after the training period.

It will be used for:

- model selection;
- configuration choices;
- feature decisions.

It will not be merged into the test data during model development.

### 4.3 Test Set

The newest period will be retained as a final hold-out test set.

This is intended to approximate the eventual production scenario:

> Train using vulnerabilities already observed and predict severity for vulnerabilities published later.

### 4.4 Why a Random Split Will Not Be the Primary Evaluation Method

A random split would allow older and newer vulnerabilities from similar publication batches to appear on both sides of the split.

This would make the test environment less representative of the intended use case and may also worsen leakage from highly templated or near-duplicate CVE descriptions.

For example, a publisher may release many similar vulnerabilities at approximately the same time.

If nearly identical descriptions appear in both training and test data, measured performance could be unrealistically high.

A chronological split is therefore more appropriate.

### 4.5 Cross-Validation

Standard random cross-validation is not planned as the primary evaluation method because it would weaken the temporal separation required by the intended production use case.

If additional model comparison is required, blocked or time-based validation may be performed within the training period.

The newest test period will remain untouched until final evaluation.

### 4.6 Evaluation Metrics

The dataset is imbalanced, particularly for the **LOW** class.

For this reason, overall accuracy alone will not be sufficient.

The primary evaluation measures will be:

- **Macro F1 score**
- **Per-class precision**
- **Per-class recall**
- **Per-class F1 score**
- **Confusion matrix**

Overall accuracy may also be reported as a secondary metric.

This approach prevents performance on the larger MEDIUM and HIGH classes from hiding poor performance on less common classes.

---

## 5. Storage Architecture and Dataset Versioning

### 5.1 Raw Data Storage

Raw source data will be stored separately from processed data and treated as immutable.

A proposed directory structure is:

```text
data/
├── raw/
│   ├── cve_list/
│   │   └── <snapshot-date>/
│   └── nvd/
│       └── <snapshot-date>/
│
├── processed/
│   └── <dataset-version>/
│
└── splits/
    └── <dataset-version>/
        ├── train.parquet
        ├── validation.parquet
        └── test.parquet
```

Keeping snapshots by date makes it possible to reproduce the dataset used by a particular training run.

### 5.2 Processed Data Storage

The processed dataset will contain a flattened, validated record for each eligible CVE.

Possible fields include:

```text
cve_id
published_at
description
cna
reference_count
validated_publication_time_metadata
nvd_cvss_v31_score
nvd_cvss_v31_severity
dataset_version
```

The target fields will be stored for training and evaluation but explicitly excluded when constructing model input features.

### 5.3 File Formats

Raw source data will remain in its original **JSON (JavaScript Object Notation)** format.

This preserves provenance and makes it possible to repeat processing from the original records.

Processed tabular datasets will preferably be stored in **Parquet** because it provides a compact typed tabular format and is more appropriate than repeatedly parsing large nested JSON files during later ML stages.

**CSV (Comma-Separated Values)** may be produced for small inspection or reporting outputs but will not be the primary processed-data format.

### 5.4 Local and Object Storage

For the initial project implementation, files can be stored locally using the same directory structure that would later be used in object storage.

A production implementation could move the same logical structure to an object store such as:

- an S3-compatible service;
- Azure Blob Storage;
- another cloud object-storage system.

Using local storage for the student implementation avoids unnecessary infrastructure complexity during Milestone 1 while still demonstrating how the architecture could later scale.

### 5.5 Dataset Versioning

Source snapshot dates and processed dataset versions will be recorded.

Git will be used for:

- source code;
- configuration;
- pipeline definitions.

A data-versioning tool such as **DVC** may later be introduced to associate larger datasets with particular code and model versions without storing the raw datasets directly inside the Git repository.

Each model training run should eventually be traceable to:

```text
Source Snapshot
      ↓
Processing Code Version
      ↓
Dataset Version
      ↓
Training Configuration
      ↓
Model Version
```

This supports reproducibility and the later MLOps requirements of the project.

### 5.6 Access Considerations

Both source datasets are public, so they do not contain private Edgescan vulnerability findings and do not require access to proprietary customer data.

Raw snapshots should be treated as read-only after ingestion so that later processing cannot silently modify the source data.

Write access to processed datasets and model artifacts should be controlled by the pipeline rather than by manual modification.

---

## 6. Proposed System Architecture

The initial end-to-end architecture is:

```text
Official CVE List                      NVD
(CVE JSON 5)                        (JSON 2.0)
       │                                  │
       └──────────────┬───────────────────┘
                      ↓
              Immutable Raw Storage
                      ↓
               Extract / Validate
                      ↓
          Join Records Using CVE ID
                      ↓
       Publication-Time Leakage Checks
                      ↓
             Clean Processed Dataset
                      ↓
         Chronological Train / Dev / Test
                      ↓
                  ML Pipeline
          TF-IDF + Metadata Features
                      ↓
            Logistic Regression
                      ↓
               Versioned Model
                      ↓
        Future Inference Service (M4)
                      ↓
          /predict    /predict/batch
```

The architecture intentionally separates raw ingestion, processing, model training and inference.

This separation allows each stage to be reproduced or replaced independently and supports the later requirement to retrain and deploy a new model when newly labelled CVEs become available.

The eventual update lifecycle will be:

```text
New CVEs Published
        ↓
Prediction Generated
        ↓
NVD Assessment Later Becomes Available
        ↓
New Labelled Examples
        ↓
Retraining
        ↓
Evaluation Against Current Model
        ↓
New Model Version
        ↓
Deployment
```

Implementation of the complete serving and retraining system belongs primarily to later milestones, particularly **Milestone 4**.

At Milestone 1, this architecture defines how the current data-source and storage decisions will support that later system.

---

## 7. Milestone 1 Summary

The proposed dataset is suitable for this project because it is sufficiently large, publicly accessible, realistic and imperfect.

The current NVD 2026 data provide more than **10,000 examples** with a consistent NVD-authored Primary CVSS v3.1 target.

The source data also contain realistic data-engineering issues such as:

- missing assessments;
- multiple scoring versions;
- conflicting assessors;
- rejected records;
- heterogeneous publishing organisations;
- records that change after publication.

The proposed ML task is a meaningful supervised multiclass classification problem.

It addresses the practical period in which a CVE is known but an NVD CVSS v3.1 assessment is not yet available.

The most important methodological constraint is temporal data leakage.

Model inputs must represent information that could realistically have been known when the prediction was made.

Current CVE records cannot automatically be treated as historical publication-time records, so publication-time reconstruction and feature validation will form an important part of the data pipeline.

A chronological training, validation and test split has been selected because it more closely represents future deployment than a random split.

Finally, the proposed storage architecture separates immutable raw snapshots from processed datasets and versioned train/validation/test data.

This provides a reproducible foundation for the automated ML pipeline, model versioning, inference service and retraining lifecycle developed in later milestones.

In [ ]:
## Milestone 2 — EDA and ETL

In [ ]:
## Milestone 3 — ML Pipeline and MLOps

In [ ]:
## Milestone 4 — Scalable Inference